# Bluestock Mutual Fund Analytics — Exploratory Data Analysis (EDA)
This notebook performs statistical and visual exploration of:
- AMC Market Share and AUM Growth (2022-2025)
- Category Allocations (Equity, Debt, Hybrid)
- Retail Monthly SIP Inflow Trajectories
- Scheme Return Correlations


In [ ]:
import os
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

print("[Notebook] Executing Exploratory Data Analysis...")
conn = sqlite3.connect('../data/db/bluestock_mf.db')
df_nav = pd.read_sql('SELECT * FROM fact_nav', conn)
df_fund = pd.read_sql('SELECT * FROM dim_fund', conn)
df_aum = pd.read_sql('SELECT * FROM fact_aum', conn)
df_sip = pd.read_csv('../data/raw/monthly_sip.csv')
conn.close()

# 1. Scheme Category Breakdown
print("[EDA] Scheme Category Count:")
print(df_fund['category'].value_counts())

# 2. AUM Concentration
print("[EDA] Total AUM by Fund House (2025):")
print(df_aum[df_aum['year'] == 2025].sort_values('aum_lakh_cr', ascending=False).head(5))

# 3. Correlation Heatmap
df_merged = df_nav.merge(df_fund, on='amfi_code')
pivot_nav = df_merged.pivot(index='date_id', columns='scheme_name', values='nav').pct_change().dropna()
top_schemes = df_fund['scheme_name'].iloc[:8]
corr_matrix = pivot_nav[top_schemes].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Return Correlation Matrix (Selected Mutual Fund Schemes)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=300)
print("[EDA] Saved correlation heatmap to ../reports/correlation_heatmap.png")
